In [ ]:
import os
# set cuda device order
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

In [ ]:
NAMES_TO_SHOW = {
    "parallel_tempering" : "NRPT", 
    "progressive_interpolation_sde" : "NRPT+ISDE",
    "smc" : "SSMC",
    "digs" : "DiGS",
    "mala" : "MALA",
    "hmc" : "HMC",
    "gmnu_2_40": "GMNU-2",
    "gm_2_40": "GM-2",
    "gmnu_16_40": "GMNU-16",
    "gm_16_40": "GM-16",
    "lennard_jones_13": "LJ-13",
    "lennard_jones_55": "LJ-55",
    "aldp_vacuum": "ALDP",
    "mlp_posterior": "BNN",
    "test/w2_data": "$W_2$",
    "test/mmd_energy": "MMD",
    "test/tvd_energy": "TVD",
    "test/w2_energy": "$W_2$ density",
    "test/relative_mae": "Relative MAE",
    "test/w2_data_equivariant": "$W_2$",
    "test/rel_mae_equivariant": "Relative MAE",
    "test/kl_div_ramachandran": "KL Ramachandran",
    "test/test_nll": "Test NLL",
    "num_density_evals": "Density evaluations",
}

In [ ]:
import torch
import numpy as np

from sampling.targets import LennardJones, ALDPVacuum, MLPPosterior
from sampling.samplers import PT

# export cuda device order and select gpu0
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

from sampling.utils.targets import (
    uniform_gaussian_mixture,
    nonuniform_gaussian_mixture
)

SAVE_PATH = "/home/fran/work_fran/sampling/experiments/figures/"

# RESULTS_PATH = "/home/fran/work_fran/sampling/experiments/notebooks/results/"
# RESULTS_PATH = "/home/fran/work_fran/sampling/experiments/notebooks/results_rebuttal/"
RESULTS_PATH = "/home/fran/work_fran/sampling/experiments/notebooks/results_arxiv/"

TASK_DICT = {
    "gm_2_40": uniform_gaussian_mixture(
        scale=40.0,
        dim=2,
        log_var_scaling=1.0,
        n_mixes=40,
        seed=0
    ),
    "gmnu_2_40": nonuniform_gaussian_mixture(
        scale=40.0,
        dim=2,
        log_var_scaling=1.0,
        n_mixes=40,
        seed=0
    ),
    "gm_16_40": uniform_gaussian_mixture(
        scale=40.0,
        dim=16,
        log_var_scaling=1.0,
        n_mixes=40,
        seed=0
    ),
    "gmnu_16_40": nonuniform_gaussian_mixture(
        scale=40.0,
        dim=16,
        log_var_scaling=1.0,
        n_mixes=40,
        seed=0
    ),
    "lennard_jones_13": LennardJones(
        data_path="/home/fran/work_fran/sampling/experiments/data/all_data_LJ13-1000.npy",
        n_particles=13,
        eps=1.0,
        rm=1.0,
        oscillator=True,
        oscillator_scale=1.0,
        energy_factor=1.0,
        interpolate=True,
        interp_min=0.65,
        interp_max=2.0,
        n_interp_points=1000            
    ),
    "lennard_jones_55": LennardJones(
        data_path="/home/fran/work_fran/sampling/experiments/data/all_data_LJ55-1000-part1.npy",
        n_particles=55,
        eps=1.0,
        rm=1.0,
        oscillator=True,
        oscillator_scale=1.0,
        energy_factor=1.0,
        interpolate=True,
        interp_min=0.65,
        interp_max=2.0,
        n_interp_points=1000            
    ),
    "aldp_vacuum": ALDPVacuum(
        data_path="/home/fran/work_fran/sampling/experiments/data/aldp_vacuum_cartesian.pt",
        temperature=300.0,
        regularize_energy=False,
        check_chirality=True,
        energy_cut=1e8,
        energy_max=1e20,
    ),
    "mlp_posterior": MLPPosterior()
}

TASK_PARAMS_DICT = {
    "gm_2_40": {
        "n_steps": 2000,
        "beta_min": 0.001,
        "n_replicas": 5,
        "step_size": 0.5,
        "n_samples": 5000,
        "time_list": [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001],
        "transport": {
            "enabled": True,
            "n_steps": 200,
            "schedule_type": "geometric",
            "noise_var": 0.001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.05,
            "compile": True,
            "verbose": False,
        },
    },
    "gmnu_2_40": {
        "n_steps": 5000,
        "beta_min": 0.001,
        "n_replicas": 5,
        "step_size": 0.5,
        "n_samples": 5000,
        "time_list": [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001],
        "transport": {
            "enabled": True,
            "n_steps": 200,
            "schedule_type": "geometric",
            "noise_var": 0.001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.05,
            "compile": True,
            "verbose": False,
        },
    },
    "gm_16_40": {
        "n_steps": 5000,
        "beta_min": 0.001,
        "n_replicas": 10,
        "step_size": 0.5,
        "n_samples": 5000,
        "time_list": np.logspace(0, -6, 7).tolist(),
        "transport": {
            "enabled": True,
            "n_steps": 300,
            "schedule_type": "geometric",
            "noise_var": 0.001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.02,
            "compile": True,
            "verbose": False,
        },
    },
    "gmnu_16_40": {
        "n_steps": 5000,
        "beta_min": 0.001,
        "n_replicas": 10,
        "step_size": 0.5,
        "n_samples": 5000,
        "time_list": [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001],
        "transport": {
            "enabled": True,
            "n_steps": 300,
            "schedule_type": "geometric",
            "noise_var": 0.001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.02,
            "compile": True,
            "verbose": False,
        },
    },
    "lennard_jones_13": {
        "n_steps": 5000,
        "beta_min": 0.8,
        "n_replicas": 3,
        "step_size": 0.001,
        "n_samples": 5000,
        "time_list": np.logspace(0, -2.5, 7).tolist(),
        "transport": {
            "enabled": True,
            "n_steps": 500,
            "schedule_type": "geometric",
            "noise_var": 0.001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.01,
            "compile": True,
            "verbose": True,
        },
    },
    "lennard_jones_55": {
        "n_steps": 5000,
        "beta_min": 0.8,
        "n_replicas": 5,
        "step_size": 0.001,
        "n_samples": 5000,
        "time_list": np.logspace(0, -2.5, 7).tolist(),
        "transport": {
            "enabled": True,
            "n_steps": 500,
            "schedule_type": "geometric",
            "noise_var": 0.001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.005,
            "compile": True,
            "verbose": True,
        },
    },
    "aldp_vacuum": {
        "n_steps": 5000,
        "beta_min": 0.3,
        "n_replicas": 3,
        "step_size": 0.000001,
        "n_samples": 10000,
        "time_list": np.logspace(0, -2.5, 7).tolist(),
        "transport": {
            "enabled": True,
            "n_steps": 500,
            "schedule_type": "geometric",
            "noise_var": 0.000001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.000001,
            "compile": True,
            "verbose": True,
        },
    },
    "mlp_posterior": {
        "n_steps": 2000,
        "beta_min": 0.8,
        "n_replicas": 5,
        "step_size": 0.00001,
        "n_samples": 2000,
        "time_list": np.logspace(0, -3, 7).tolist(),
        "transport": {
            "enabled": True,
            "n_steps": 500,
            "schedule_type": "geometric",
            "noise_var": 0.00001,
            "corrector_mode": "mala",
            "corrector_steps": 1,
            "corrector_step_size": 0.00001,
            "compile": True,
            "verbose": True,
        },
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# STEP_SIZE = 0.0002
# STEP_SIZE = 0.01 # GB=3.algo



In [ ]:
def build_interpolation_schedule(
    type: str, 
    n_steps: int,
    min_val: float = 0.0,
    max_val: float = 1.0,
    p: float = 1.0
):
    if type == "linear":
        schedule = torch.linspace(
            min_val,
            max_val,
            n_steps
        )
    elif type == "polynomial":
        schedule = torch.linspace(
            min_val,
            max_val,
            n_steps
        )
        schedule = min_val + (max_val - min_val) * (schedule ** p)
    elif type == "geometric":
        r = (max_val / min_val) ** (1.0 / (n_steps - 1))
        schedule = min_val * (r ** torch.arange(n_steps))
    else:
        raise ValueError(f"Unknown schedule type: {type}")

    schedule = torch.clamp(schedule, min=min_val, max=max_val)
    
    return schedule

In [ ]:
# task_name = "gm_16_40"
task_name = "lennard_jones_13"
# task_name = "mlp_posterior"
# task_name = "aldp_vacuum"
target = TASK_DICT[task_name]

n_samples   = TASK_PARAMS_DICT[task_name]["n_samples"]  
n_steps = TASK_PARAMS_DICT[task_name]["n_steps"]
beta_min = TASK_PARAMS_DICT[task_name]["beta_min"]
n_replicas = TASK_PARAMS_DICT[task_name]["n_replicas"]
step_size = TASK_PARAMS_DICT[task_name]["step_size"]

gt_samples = target.sample(n_samples).to(DEVICE)

if task_name in ["lennard_jones_13", "lennard_jones_55", "aldp_vacuum"]:
    init_point = torch.load(f"/home/fran/work_fran/sampling/experiments/data/{task_name}_minenergy.pt").to(DEVICE)
elif task_name == "mlp_posterior":
    init_point = torch.load(f"/home/fran/work_fran/sampling/experiments/data/mlp_posterior_sample_prior.pt").to(DEVICE)
else:
    init_point = torch.randn(1, target.dim).to(DEVICE)
beta_schedule = build_interpolation_schedule(
    type="geometric",
    n_steps=n_replicas,
    min_val=beta_min,
    max_val=1.0
).to(DEVICE)

In [ ]:
from sampling.utils.metrics import wasserstein2_distance, kl_div_ramachandran

from sampling.targets import Gaussian, Uniform, ConditionalTarget, InfUniform
from sampling.kernels import MALAKernel, HMCKernel
from sampling.samplers import ProgressiveInterpolationSDE, PT

import matplotlib.pyplot as plt

import mdtraj
from openmmtools.testsystems import AlanineDipeptideVacuum


n_folds = 1

# Post-PT transport settings (read from TASK_PARAMS_DICT)
transport_cfg = TASK_PARAMS_DICT[task_name].get("transport")
USE_POST_PT_TRANSPORT = transport_cfg.get("enabled")
TRANSPORT_N_STEPS = transport_cfg.get("n_steps")
TRANSPORT_SCHEDULE_TYPE = transport_cfg.get("schedule_type")
TRANSPORT_NOISE_VAR = transport_cfg.get("noise_var")
TRANSPORT_CORRECTOR_MODE = transport_cfg.get("corrector_mode")
TRANSPORT_CORRECTOR_STEPS = transport_cfg.get("corrector_steps")
TRANSPORT_CORRECTOR_STEP_SIZE = transport_cfg.get("corrector_step_size")
TRANSPORT_COMPILE = transport_cfg.get("compile")
TRANSPORT_VERBOSE = transport_cfg.get("verbose")

result_suffix = "cond_pt_plus_pisde" if USE_POST_PT_TRANSPORT else "cond_pt"
results_filename = f"{RESULTS_PATH}/{task_name}_{result_suffix}_timeinits_{n_steps}.npz"

time_list = TASK_PARAMS_DICT[task_name].get(
    "time_list",
)
rt_counts_list = []
samples_list = []
rejection_rates_list = []
perf_metric_list = []

for time_init in time_list:

    for _ in range(n_folds):

        print("----------------------------------------")
        print(f"t0: {time_init}")

        z = init_point.repeat(n_samples, 1).to(DEVICE)
        # z = torch.randn(N_SAMPLES, target.dim).to(DEVICE)

        z_replicas = z.unsqueeze(1).repeat(1, n_replicas, 1) # (n_samples, n_replicas, dim)
        z_replicas_flat = z_replicas.reshape(-1, target.dim) # (n_samples * n_replicas, dim)

        if task_name == "aldp_vacuum":
            kernel = HMCKernel(
                n_leapfrog_steps=5,
                momentum_scale=time_init,
            )
        else: 
            kernel = MALAKernel(noise_scale=time_init)
        cond_target = ConditionalTarget(
            target=target,
            time=time_init,
            z=z_replicas_flat
        )

        # reference = InfUniform(dim=target.dim).to(DEVICE)
        reference = Gaussian(
            mean=z.mean(dim=0),
            std=torch.ones_like(z.mean(dim=0))
        ).to(DEVICE)
        sampler = PT(
            target=cond_target,
            kernel=kernel,
            reference=reference,
            beta_schedule=beta_schedule,
            step_size=step_size*time_init,
            swap_mode="nrpt",
            adaptation_rate=0.05,
            swap_every=1,
            verbose=True,
            compile=True

        )
        sampler = sampler.to(DEVICE)

        x0 = z.clone()
        # x0 = (1.0-time_init) * z + time_init * gt_samples.clone()

        samples, log_acceptance, rt_counts = sampler(
            x0,
            n_steps=n_steps,
            return_log_acceptance=True,
            return_round_trip_counts=True
        )         # samples: (n_samples, n_replicas, dim), log_acceptance: (n_steps, n_samples, n_replicas-1)

        samples = samples[:, -1, :] # (n_samples, dim)

        if USE_POST_PT_TRANSPORT:
            transport_time_schedule = build_interpolation_schedule(
                type=TRANSPORT_SCHEDULE_TYPE,
                n_steps=TRANSPORT_N_STEPS,
                min_val=time_init,
                max_val=1.0
            ).to(DEVICE)
            transport_noise_schedule = torch.full_like(
                transport_time_schedule,
                fill_value=TRANSPORT_NOISE_VAR
            )
            transport_sampler = ProgressiveInterpolationSDE(
                target=target,
                time_schedule=transport_time_schedule,
                noise_schedule=transport_noise_schedule,
                corrector_mode=TRANSPORT_CORRECTOR_MODE,
                corrector_steps=TRANSPORT_CORRECTOR_STEPS,
                corrector_step_size=TRANSPORT_CORRECTOR_STEP_SIZE,
                jump_steps=0,
                compile=TRANSPORT_COMPILE,
                verbose=TRANSPORT_VERBOSE
            ).to(DEVICE)
            samples = transport_sampler(
                x0=samples,
                z=z
            )

        acc_probs = torch.exp(torch.clamp(log_acceptance, max=0.0)) # (n_replicas-1,)
        rejection_rates = 1.0 - acc_probs  # (n_replicas-1,)
        
        if task_name in ["mlp_posterior"]:
            # for the MLP posterior, we can compute the true W2 distance in closed form since it's a Gaussian
            test_nll = - target.log_likelihood(target.X_test, target.y_test, samples).mean() / target.X_test.shape[0]
            print(f"Test NLL: {test_nll:.4f}")
            perf_metric = test_nll
        elif task_name in ["aldp_vacuum"]:

            n_particles = 22
            dim = samples.shape[1]

            aldp_system = AlanineDipeptideVacuum(constraints=None)

            topology = mdtraj.Topology.from_openmm(aldp_system.topology)

            gt_samples_np = gt_samples.view(-1, n_particles, dim // n_particles).cpu().numpy()
            pred_samples_np = samples.view(-1, n_particles, dim // n_particles).cpu().numpy()

            traj_gt = mdtraj.Trajectory(gt_samples_np, topology)
            phi_gt = mdtraj.compute_phi(traj_gt)[1].reshape(-1)
            psi_gt = mdtraj.compute_psi(traj_gt)[1].reshape(-1)
            not_nan_idx = ~np.isnan(phi_gt) & ~np.isnan(psi_gt)
            phi_gt = phi_gt[not_nan_idx]
            psi_gt = psi_gt[not_nan_idx]

            traj_pred = mdtraj.Trajectory(pred_samples_np, topology)
            phi_pred = mdtraj.compute_phi(traj_pred)[1].reshape(-1)
            psi_pred = mdtraj.compute_psi(traj_pred)[1].reshape(-1)
            not_nan_idx = ~np.isnan(phi_pred) & ~np.isnan(psi_pred)
            phi_pred = phi_pred[not_nan_idx]
            psi_pred = psi_pred[not_nan_idx]

            kl_div_ram = kl_div_ramachandran(
                phi_gt=torch.tensor(phi_gt, device=DEVICE),
                psi_gt=torch.tensor(psi_gt, device=DEVICE),
                phi_pred=torch.tensor(phi_pred, device=DEVICE),
                psi_pred=torch.tensor(psi_pred, device=DEVICE),
                num_bins=100,
                eps=1e-10
            )
            perf_metric = kl_div_ram
            print(f"KL divergence: {kl_div_ram:.4f}")
        else:
            w2 = wasserstein2_distance(
                gt_samples.detach(),
                samples.detach()
            )
            print(f"W2: {w2:.4f}")
            perf_metric = w2

        print(f"Round-trip counts: {rt_counts.sum(dim=1).mean(dtype=torch.float32).cpu()}")

        rt_counts_list.append(rt_counts.cpu().numpy())
        samples_list.append(samples.detach())
        rejection_rates_list.append(rejection_rates.cpu().numpy())
        perf_metric_list.append(perf_metric.cpu().numpy())

# save results to file
np.savez(
    results_filename,
    times=np.array(time_list),
    rt_counts=np.array(rt_counts_list),
    samples=np.array([s.cpu().numpy() for s in samples_list]),
    perf_metric=np.array(perf_metric_list),
    rejection_rates=np.array(rejection_rates_list)
)


In [ ]:

# read results from file
data = np.load(results_filename, allow_pickle=True)
time_list = data["times"]
rt_counts_list = data["rt_counts"]
samples_list = data["samples"]
rejection_rates_list = data["rejection_rates"]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# change text size globally
plt.rcParams.update({'font.size': 16})

# Assuming time_list and rejection_rates_list are already defined

n_lines = len(time_list)

# 1. Create a color gradient (e.g., using 'viridis', 'plasma', or 'coolwarm')
# We generate 'n_lines' colors evenly spaced from 0 to 1
# colors = cm.viridis(np.linspace(0, 1, n_lines))
colors = cm.Blues(np.linspace(0.3, 1, n_lines))

# 2. Create a single plot
fig, ax = plt.subplots(figsize=(7, 5))

for i in range(n_lines):
    # Preserve your original reverse-order logic
    idx = i
    time_init = time_list[idx]
    rejection_rates = rejection_rates_list[idx] # (n_samples, n_replicas-1)
    
    rejection_rates_mean = rejection_rates.mean(axis=0)
    rejection_rates_std = rejection_rates.std(axis=0)


    # 3. Assign the color based on the index so the gradient matches the time order
    color = colors[idx]

    time_exponent = int(np.log10(time_init))
    
    ax.plot(rejection_rates_mean, marker='s', color=color, lw=2.5, ms=8, label=r"$t_0 = 10^{" + str(time_exponent) + r"}$")
    # ax.fill_between(
    #     np.arange(len(rejection_rates_mean)),
    #     rejection_rates_mean - rejection_rates_std,
    #     rejection_rates_mean + rejection_rates_std,
    #     color=color,
    #     alpha=0.3
    # )

# Formatting
ax.set_ylim(-0.1, 1.1)
ax.set_ylabel("Rejection rate")

# Set x-ticks (assuming all lists have same length, taking the last one processed)
ax.set_xticks(np.arange(n_replicas-1))
ax.set_xticklabels([fr"{j+1}$\leftrightarrow${j+2}" for j in range(n_replicas-1)])
# modify font size of x-tick labels
ax.set_xlabel("Replica pairs")
ax.tick_params(axis='x', which='major', labelsize=14)

ax.grid(visible=True, which='both', linestyle='--', linewidth=0.5)

# Add a legend to replace the titles
ax.legend(frameon=False, fontsize=12)

ax.set_title(f"{NAMES_TO_SHOW[task_name]}")

plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/rejection_rates_{task_name}_{n_steps}.pdf", bbox_inches='tight')
plt.show()


In [ ]:
if task_name == "mlp_posterior":
    gcb_std_mult = 0.1
    rtc_std_mult = 0.2
else:
    gcb_std_mult = 1.0
    rtc_std_mult = 1.0


In [ ]:
# Combined plot

# Round trip rate and global communication barrier plot
rtc_mean_list = []
rtc_std_list = []
for rt_counts in rt_counts_list:
    rt_counts = rt_counts.sum(axis=1) # (n_samples,)
    rtc_mean_list.append(rt_counts.mean())
    rtc_std_list.append(rt_counts.std())

gcb_mean_list = []
gcb_std_list = []
for i in range(len(time_list)):
    rejection_rates = rejection_rates_list[i] # (n_samples, n_replicas-1)
    Lambda = np.cumsum(rejection_rates, axis=1) # (n_samples, n_replicas-1)
    gbc_mean = Lambda[:,-1].mean()
    gcb_std = Lambda[:,-1].std()
    gcb_mean_list.append(gbc_mean)
    gcb_std_list.append(gcb_std)

# 2. Create the plot
fig, ax1 = plt.subplots(figsize=(5, 4))

# --- Plot 1: Round Trip Rate (Left Axis) ---
color1 = 'darkblue'
ax1.set_xlabel("$t_0$")
ax1.set_ylabel("Round trips", color=color1)
ax1.plot(time_list, rtc_mean_list, color=color1, marker='s', lw=2.5, markersize=8, label="Round Trip")
ax1.fill_between(
    time_list,
    np.array(rtc_mean_list) - rtc_std_mult*np.array(rtc_std_list),
    np.array(rtc_mean_list) + rtc_std_mult*np.array(rtc_std_list),
    color=color1,
    alpha=0.2
)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xscale("log")

# --- Plot 2: Global Comm Barrier (Right Axis) ---
ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color2 = 'firebrick'
ax2.set_ylabel(r"Global com. barrier", color=color2)
ax2.plot(time_list, gcb_mean_list, color=color2, marker='s', lw=2.5, markersize=8, label="Global Barrier")
ax2.fill_between(
    time_list,
    np.array(gcb_mean_list) - gcb_std_mult*np.array(gcb_std_list),
    np.array(gcb_mean_list) + gcb_std_mult*np.array(gcb_std_list),
    color=color2,
    alpha=0.2
)

ax2.tick_params(axis='y', labelcolor=color2)

# Optional: Add a combined legend (handling two axes)
# lines_1, labels_1 = ax1.get_legend_handles_labels()
# lines_2, labels_2 = ax2.get_legend_handles_labels()
# ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc=0)

# add grid
ax1.grid(visible=True, which='both', linestyle='--', linewidth=0.5)

ax1.set_title(f"{NAMES_TO_SHOW[task_name]}")


plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.15)
# plt.savefig(f"{SAVE_PATH}/rtc_gcb_{task_name}_{n_steps}.pdf")
plt.show()


In [ ]:
# Separate plot: RTR

fig, ax = plt.subplots(figsize=(4, 4))
ax.set_xlabel("$t_0$")
ax.set_ylabel("Round trips")
ax.plot(time_list, rtc_mean_list, color='darkblue', marker='s', lw=2.5, markersize=8)
ax.fill_between(
    time_list,
    np.array(rtc_mean_list) - rtc_std_mult*np.array(rtc_std_list),
    np.array(rtc_mean_list) + rtc_std_mult*np.array(rtc_std_list),
    color='darkblue',
    alpha=0.2
)
ax.set_xscale("log")
ax.grid(visible=True, which='both', linestyle='--', linewidth=0.5)
ax.set_title(f"{NAMES_TO_SHOW[task_name]}")
plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.15)
plt.savefig(f"{SAVE_PATH}/rtc_{task_name}_{n_steps}.pdf")

In [ ]:
# separate plot: GCB

fig, ax = plt.subplots(figsize=(4, 4))
ax.set_xlabel("$t_0$")
ax.set_ylabel(r"Global com. barrier")
ax.plot(time_list, gcb_mean_list, color='firebrick', marker='s', lw=2.5, markersize=8)
ax.fill_between(
    time_list,
    np.array(gcb_mean_list) - gcb_std_mult*np.array(gcb_std_list),
    np.array(gcb_mean_list) + gcb_std_mult*np.array(gcb_std_list),
    color='firebrick',
    alpha=0.2
)
ax.set_xscale("log")
ax.grid(visible=True, which='both', linestyle='--', linewidth=0.5)
ax.set_title(f"{NAMES_TO_SHOW[task_name]}")
plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.15)
plt.savefig(f"{SAVE_PATH}/gcb_{task_name}_{n_steps}.pdf")
plt.show()

In [ ]:


if target.dim == 2:
    # Plot samples
    n_cols = len(time_list)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    for i in range(len(time_list)):
        ax = axes[i] if n_cols > 1 else axes

        time_init = time_list[len(time_list)-1 - i]
        samples = samples_list[len(time_list)-1 - i]

        gt_samples_scaled = (1-time_init) * z + time_init * gt_samples.clone()

        ax.scatter(
            gt_samples_scaled.cpu().numpy()[:, 0],
            gt_samples_scaled.cpu().numpy()[:, 1],
            color="blue",
            alpha=0.5,
            label="GT Samples",
            s=5
        )

        ax.scatter(
            samples.cpu().numpy()[:, 0],
            samples.cpu().numpy()[:, 1],
            color="red",
            alpha=0.5,
            label="PI SDE Samples",
            s=5
        )
        ax.set_title(f"t0: {time_init}")
    plt.tight_layout()
    plt.show()